# Notebook 28 — T4c: kiểm chứng trên ba input 5 task × 3 host

Câu hỏi: lợi ích của trạng thái đầu/mixer domain-wall trên development N27 có xuất hiện trên ba input đã khóa từ N26 không?

Giữ nguyên ba nhánh, exact mean-energy + COBYLA, p1→p2, penalty4. Mỗi nhánh/case cap100 p1 +100 p2; 18stage, tổng cap1800 objective. Mọi case đều25qubit. Không chọn lại dữ liệu, tăng cap hay đổi phương pháp sau khi xem kết quả.

N27 đã nghiệm thu: sáu stage hoàn tất,570/600objective. Giữ nguyên thư mục kết quả N27 và N27_ACTIVE.json. N28 kiểm tiền đề này trước khi tạo lượt chạy chính. Mã mới nằm trong experiments/, không thay đổi src/ đã khóa của N27.

Chạy lần lượt; mỗi ô huấn luyện hoàn tất p1+p2 của một nhánh. Có thể nghỉ sau khi ô đó kết thúc. Đọc N28_README_FIRST.md trước khi chạy. Không đổi môi trường/package giữa campaign.

## 1. Mở đúng repo và kernel nt409-khiem


In [1]:
from pathlib import Path
import sys
import psutil
from IPython.display import display, Markdown
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'experiments/domain_wall_validation.py').exists()), None)
if root is None:
    raise RuntimeError('Giải nén gói N28 vào repo đã chạy N27 rồi mở notebook tại đó.')
if str(root) not in sys.path: sys.path.insert(0, str(root))
from experiments.domain_wall_validation import open_campaign, checked_campaign, run_campaign, summarize, export_campaign
print('Repo:', root)
print('Available RAM:', round(psutil.virtual_memory().available/2**30, 2), 'GiB')
print('5 tasks x 3 hosts / case | 25 qubits | Aer CPU 4 threads | RAM guard 4 GiB')

def show_status(status, instance_id=None):
    rows=[r for r in status['rows'] if instance_id is None or r['instance_id']==instance_id]
    lines=['| Case | Nhánh | p | Trạng thái | nfev/cap | E | P_domain % | P_valid % | P_opt % | P_ground |',
           '|---|---|---:|---|---:|---:|---:|---:|---:|---:|']
    for r in rows:
        metric=lambda key: f"{100*r[key]:.6f}" if key in r else '-'
        energy=f"{r['mean_energy']:.6f}" if 'mean_energy' in r else '-'
        ground=f"{r['p_ground']:.6g}" if 'p_ground' in r else '-'
        lines.append(f"| {r['instance_id']} | {r['variant']} | {r['p']} | {r['status']} | {r.get('objective_calls_completed',0)}/{r['cap']} | {energy} | {metric('p_domain')} | {metric('p_valid')} | {metric('p_opt')} | {ground} |")
    display(Markdown('\n'.join(lines)))
    print('Completed:',status['completed_stages'],'/18 | objective:',status['objective_calls_reserved'],'/',status['objective_cap'])


Repo: c:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement
Available RAM: 11.9 GiB
5 tasks x 3 hosts / case | 25 qubits | Aer CPU 4 threads | RAM guard 4 GiB


## 2. Kiểm tra N27, dữ liệu và ngân sách

Ô này chưa huấn luyện QAOA. Nó tạo hoặc mở lại đúng một campaign N28, khóa input/source/runtime và ghi liên kết với N27. Lần đầu phải thấy `software_validation=false`, ba case, mỗi case25qubit,18stage và cap1800. Nếu N27 chưa hoàn tất, source/runtime khác hoặc dữ liệu bị sửa, dừng để kiểm tra; không bỏ guard.


In [2]:
folder = open_campaign(root)
campaign = checked_campaign(root, folder)
assert campaign['identity']['software_validation'] is False
assert len(campaign['plan']) == 18
assert sum(s['cap'] for s in campaign['plan']) == 1800
assert [x['input']['instance_id'] for x in campaign['identity']['cases']] == ['v5x3_1','v5x3_2','v5x3_3']
assert all(x['qubits']==25 for x in campaign['identity']['cases'])
print('CAMPAIGN:',folder)
print('software_validation:',campaign['identity']['software_validation'])
print('N27 verified:',campaign['identity']['predecessor'])
for case in campaign['identity']['cases']:
    print(case['input']['instance_id'],'| qubits',case['qubits'],'| cap 600 | input SHA256',case['input_sha256'])
print('Stages: 18 | total objective cap: 1800')
show_status(summarize(root,folder))


CAMPAIGN: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_validation\t4c_20261026T131859190531Z
software_validation: False
N27 verified: {'relative_folder': 'data/reference_results/phase5_domainwall_training/t4b_20261025T070339188480Z', 'campaign_sha256': 'a47d964274053ba0580d1fd0577b6d40f661aacd4e0647a41c2c7a6f974b8b9b', 'identity_sha256': '8087fbb7511535ec36c398970d98561d24cd196c408f5233f105f17dbc5a2fce', 'completed_stages': 6, 'objective_calls': 570}
v5x3_1 | qubits 25 | cap 600 | input SHA256 e3f4ba0ce5b66a51077619867d4f120603074a52043de9c980435f9d0b768a70
v5x3_2 | qubits 25 | cap 600 | input SHA256 6598feadf65a484d512329a6112944496771cec177fd4794bb54735a1b999a68
v5x3_3 | qubits 25 | cap 600 | input SHA256 dacf65f766d768c3b54db00d1e55b0254ff2b6575537b9fdecb9381d0f2d7847
Stages: 18 | total objective cap: 1800


| Case | Nhánh | p | Trạng thái | nfev/cap | E | P_domain % | P_valid % | P_opt % | P_ground |
|---|---|---:|---|---:|---:|---:|---:|---:|---:|
| v5x3_1 | standard_x | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_1 | standard_x | 2 | pending | 0/100 | - | - | - | - | - |
| v5x3_1 | valid_x | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_1 | valid_x | 2 | pending | 0/100 | - | - | - | - | - |
| v5x3_1 | valid_dw | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_1 | valid_dw | 2 | pending | 0/100 | - | - | - | - | - |
| v5x3_2 | standard_x | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_2 | standard_x | 2 | pending | 0/100 | - | - | - | - | - |
| v5x3_2 | valid_x | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_2 | valid_x | 2 | pending | 0/100 | - | - | - | - | - |
| v5x3_2 | valid_dw | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_2 | valid_dw | 2 | pending | 0/100 | - | - | - | - | - |
| v5x3_3 | standard_x | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_3 | standard_x | 2 | pending | 0/100 | - | - | - | - | - |
| v5x3_3 | valid_x | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_3 | valid_x | 2 | pending | 0/100 | - | - | - | - | - |
| v5x3_3 | valid_dw | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_3 | valid_dw | 2 | pending | 0/100 | - | - | - | - | - |

Completed: 0 /18 | objective: 0 / 1800


## Chạy v5x3_1

Oracle có 78/243 assignment khả thi; optimum 2 host. Đây là thông tin đã khóa từ N26, chỉ dùng chấm điểm, không dùng chọn góc. Tách P_valid và P_opt: placement khả thi có thể dùng 2 hoặc 3 host.

Chạy ba ô dưới theo thứ tự. Mỗi ô giữ p1 và p2 cùng một nhánh; cap200. Log xuất hiện ở eval1 và mỗi10lượt; trước eval1 phải dựng mô hình/diagonal/mạch nên có thể chờ hàng chục giây.


### v5x3_1 — standard_x: p1 → p2


In [3]:
status = run_campaign(root, folder, instance_id='v5x3_1', variant='standard_x')
show_status(status, 'v5x3_1')
archive = export_campaign(root,folder)
print('Checkpoint ZIP:',archive)
selected = [r for r in status['rows'] if r['instance_id']=='v5x3_1' and r['variant']=='standard_x']
assert len(selected)==2 and all(r['status']=='completed' for r in selected), 'Stage chưa hoàn tất. Giữ ZIP/log để kiểm tra; không reset ngân sách.'
print('v5x3_1 / standard_x: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.')


Run v5x3_1__standard_x_p1 | 25 qubits | cap 100
  eval 1/100 | E=550.249686
  eval 10/100 | E=220.817400
  eval 20/100 | E=208.405193
  eval 30/100 | E=186.269650
  eval 40/100 | E=99.543270
  eval 50/100 | E=98.453774
  eval 60/100 | E=98.318164
Run v5x3_1__standard_x_p2 | 25 qubits | cap 100
  eval 1/100 | E=117.274671
  eval 10/100 | E=86.807252
  eval 20/100 | E=63.336273
  eval 30/100 | E=61.411932
  eval 40/100 | E=61.100029
  eval 50/100 | E=60.670964
  eval 60/100 | E=59.974574
  eval 70/100 | E=59.514442
  eval 80/100 | E=59.310407
  eval 90/100 | E=59.222998
  eval 100/100 | E=59.087622


| Case | Nhánh | p | Trạng thái | nfev/cap | E | P_domain % | P_valid % | P_opt % | P_ground |
|---|---|---:|---|---:|---:|---:|---:|---:|---:|
| v5x3_1 | standard_x | 1 | completed | 66/100 | 98.317334 | 45.754794 | 5.434750 | 0.207156 | 1.33952e-06 |
| v5x3_1 | standard_x | 2 | completed | 100/100 | 59.087622 | 48.518063 | 11.566711 | 0.278349 | 1.2236e-05 |
| v5x3_1 | valid_x | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_1 | valid_x | 2 | pending | 0/100 | - | - | - | - | - |
| v5x3_1 | valid_dw | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_1 | valid_dw | 2 | pending | 0/100 | - | - | - | - | - |

Completed: 2 /18 | objective: 166 / 1800
Checkpoint ZIP: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_validation\t4c_20261026T131859190531Z.zip
v5x3_1 / standard_x: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.


### v5x3_1 — valid_x: p1 → p2


In [4]:
status = run_campaign(root, folder, instance_id='v5x3_1', variant='valid_x')
show_status(status, 'v5x3_1')
archive = export_campaign(root,folder)
print('Checkpoint ZIP:',archive)
selected = [r for r in status['rows'] if r['instance_id']=='v5x3_1' and r['variant']=='valid_x']
assert len(selected)==2 and all(r['status']=='completed' for r in selected), 'Stage chưa hoàn tất. Giữ ZIP/log để kiểm tra; không reset ngân sách.'
print('v5x3_1 / valid_x: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.')


Run v5x3_1__valid_x_p1 | 25 qubits | cap 100
  eval 1/100 | E=352.776538
  eval 10/100 | E=180.120806
  eval 20/100 | E=179.498481
  eval 30/100 | E=179.240973
  eval 40/100 | E=178.562007
  eval 50/100 | E=178.175415
  eval 60/100 | E=175.708350
  eval 70/100 | E=168.021934
  eval 80/100 | E=165.460381
  eval 90/100 | E=155.038575
  eval 100/100 | E=150.516848
Run v5x3_1__valid_x_p2 | 25 qubits | cap 100
  eval 1/100 | E=159.012892
  eval 10/100 | E=166.595656
  eval 20/100 | E=143.664103
  eval 30/100 | E=138.085386
  eval 40/100 | E=123.499166
  eval 50/100 | E=116.791970
  eval 60/100 | E=115.202452
  eval 70/100 | E=99.167761
  eval 80/100 | E=88.917162
  eval 90/100 | E=84.277732
  eval 100/100 | E=77.440332


| Case | Nhánh | p | Trạng thái | nfev/cap | E | P_domain % | P_valid % | P_opt % | P_ground |
|---|---|---:|---|---:|---:|---:|---:|---:|---:|
| v5x3_1 | standard_x | 1 | completed | 66/100 | 98.317334 | 45.754794 | 5.434750 | 0.207156 | 1.33952e-06 |
| v5x3_1 | standard_x | 2 | completed | 100/100 | 59.087622 | 48.518063 | 11.566711 | 0.278349 | 1.2236e-05 |
| v5x3_1 | valid_x | 1 | completed | 100/100 | 150.516848 | 72.665298 | 3.799145 | 0.297157 | 1.03636e-05 |
| v5x3_1 | valid_x | 2 | completed | 100/100 | 77.440332 | 43.550876 | 7.296463 | 0.260118 | 1.87387e-05 |
| v5x3_1 | valid_dw | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_1 | valid_dw | 2 | pending | 0/100 | - | - | - | - | - |

Completed: 4 /18 | objective: 366 / 1800
Checkpoint ZIP: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_validation\t4c_20261026T131859190531Z.zip
v5x3_1 / valid_x: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.


### v5x3_1 — valid_dw: p1 → p2


In [5]:
status = run_campaign(root, folder, instance_id='v5x3_1', variant='valid_dw')
show_status(status, 'v5x3_1')
archive = export_campaign(root,folder)
print('Checkpoint ZIP:',archive)
selected = [r for r in status['rows'] if r['instance_id']=='v5x3_1' and r['variant']=='valid_dw']
assert len(selected)==2 and all(r['status']=='completed' for r in selected), 'Stage chưa hoàn tất. Giữ ZIP/log để kiểm tra; không reset ngân sách.'
print('v5x3_1 / valid_dw: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.')


Run v5x3_1__valid_dw_p1 | 25 qubits | cap 100
  eval 1/100 | E=334.730481
  eval 10/100 | E=147.712864
  eval 20/100 | E=158.037136
  eval 30/100 | E=99.002291
  eval 40/100 | E=98.814967
  eval 50/100 | E=98.814716
Run v5x3_1__valid_dw_p2 | 25 qubits | cap 100
  eval 1/100 | E=98.604792
  eval 10/100 | E=91.415475
  eval 20/100 | E=71.552868
  eval 30/100 | E=68.748541
  eval 40/100 | E=68.680067
  eval 50/100 | E=68.664095
  eval 60/100 | E=68.655820
  eval 70/100 | E=68.650457
  eval 80/100 | E=68.643134
  eval 90/100 | E=68.640808
  eval 100/100 | E=68.637130


| Case | Nhánh | p | Trạng thái | nfev/cap | E | P_domain % | P_valid % | P_opt % | P_ground |
|---|---|---:|---|---:|---:|---:|---:|---:|---:|
| v5x3_1 | standard_x | 1 | completed | 66/100 | 98.317334 | 45.754794 | 5.434750 | 0.207156 | 1.33952e-06 |
| v5x3_1 | standard_x | 2 | completed | 100/100 | 59.087622 | 48.518063 | 11.566711 | 0.278349 | 1.2236e-05 |
| v5x3_1 | valid_x | 1 | completed | 100/100 | 150.516848 | 72.665298 | 3.799145 | 0.297157 | 1.03636e-05 |
| v5x3_1 | valid_x | 2 | completed | 100/100 | 77.440332 | 43.550876 | 7.296463 | 0.260118 | 1.87387e-05 |
| v5x3_1 | valid_dw | 1 | completed | 53/100 | 98.814708 | 100.000000 | 13.568403 | 0.986681 | 1.90717e-05 |
| v5x3_1 | valid_dw | 2 | completed | 100/100 | 68.636648 | 100.000000 | 21.618602 | 0.848718 | 2.79811e-05 |

Completed: 6 /18 | objective: 519 / 1800
Checkpoint ZIP: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_validation\t4c_20261026T131859190531Z.zip
v5x3_1 / valid_dw: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.


## Chạy v5x3_2

Oracle có 1/243 assignment khả thi; optimum 3 host. Đây là thông tin đã khóa từ N26, chỉ dùng chấm điểm, không dùng chọn góc. Optimum=3 nên P_opt=P_valid; zero optimal shots không chứng minh bài toán vô nghiệm.

Chạy ba ô dưới theo thứ tự. Mỗi ô giữ p1 và p2 cùng một nhánh; cap200. Log xuất hiện ở eval1 và mỗi10lượt; trước eval1 phải dựng mô hình/diagonal/mạch nên có thể chờ hàng chục giây.


### v5x3_2 — standard_x: p1 → p2


In [6]:
status = run_campaign(root, folder, instance_id='v5x3_2', variant='standard_x')
show_status(status, 'v5x3_2')
archive = export_campaign(root,folder)
print('Checkpoint ZIP:',archive)
selected = [r for r in status['rows'] if r['instance_id']=='v5x3_2' and r['variant']=='standard_x']
assert len(selected)==2 and all(r['status']=='completed' for r in selected), 'Stage chưa hoàn tất. Giữ ZIP/log để kiểm tra; không reset ngân sách.'
print('v5x3_2 / standard_x: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.')


Run v5x3_2__standard_x_p1 | 25 qubits | cap 100
  eval 1/100 | E=685.748557
  eval 10/100 | E=314.178297
  eval 20/100 | E=310.191189
  eval 30/100 | E=309.680118
  eval 40/100 | E=309.593401
  eval 50/100 | E=309.576640
  eval 60/100 | E=309.565111
  eval 70/100 | E=309.559506
  eval 80/100 | E=309.554824
  eval 90/100 | E=309.554771
  eval 100/100 | E=309.551924
Run v5x3_2__standard_x_p2 | 25 qubits | cap 100
  eval 1/100 | E=326.175909
  eval 10/100 | E=313.703800
  eval 20/100 | E=306.308381
  eval 30/100 | E=305.643886
  eval 40/100 | E=304.849903
  eval 50/100 | E=301.358519
  eval 60/100 | E=295.553431
  eval 70/100 | E=291.984476
  eval 80/100 | E=289.435005
  eval 90/100 | E=287.521642
  eval 100/100 | E=286.110019


| Case | Nhánh | p | Trạng thái | nfev/cap | E | P_domain % | P_valid % | P_opt % | P_ground |
|---|---|---:|---|---:|---:|---:|---:|---:|---:|
| v5x3_2 | standard_x | 1 | completed | 100/100 | 309.551864 | 29.478311 | 0.019687 | 0.019687 | 1.25052e-06 |
| v5x3_2 | standard_x | 2 | completed | 100/100 | 286.110019 | 35.374913 | 0.036670 | 0.036670 | 7.31844e-06 |
| v5x3_2 | valid_x | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_2 | valid_x | 2 | pending | 0/100 | - | - | - | - | - |
| v5x3_2 | valid_dw | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_2 | valid_dw | 2 | pending | 0/100 | - | - | - | - | - |

Completed: 8 /18 | objective: 719 / 1800
Checkpoint ZIP: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_validation\t4c_20261026T131859190531Z.zip
v5x3_2 / standard_x: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.


### v5x3_2 — valid_x: p1 → p2


In [7]:
status = run_campaign(root, folder, instance_id='v5x3_2', variant='valid_x')
show_status(status, 'v5x3_2')
archive = export_campaign(root,folder)
print('Checkpoint ZIP:',archive)
selected = [r for r in status['rows'] if r['instance_id']=='v5x3_2' and r['variant']=='valid_x']
assert len(selected)==2 and all(r['status']=='completed' for r in selected), 'Stage chưa hoàn tất. Giữ ZIP/log để kiểm tra; không reset ngân sách.'
print('v5x3_2 / valid_x: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.')


Run v5x3_2__valid_x_p1 | 25 qubits | cap 100
  eval 1/100 | E=469.963093
  eval 10/100 | E=234.434642
  eval 20/100 | E=226.105540
  eval 30/100 | E=221.323710
  eval 40/100 | E=197.477038
  eval 50/100 | E=168.151941
  eval 60/100 | E=156.402020
  eval 70/100 | E=154.189576
  eval 80/100 | E=153.659423
  eval 90/100 | E=153.651899
  eval 100/100 | E=153.650814
Run v5x3_2__valid_x_p2 | 25 qubits | cap 100
  eval 1/100 | E=134.645517
  eval 10/100 | E=154.170624
  eval 20/100 | E=108.031866
  eval 30/100 | E=105.912201
  eval 40/100 | E=105.413867
  eval 50/100 | E=105.409981
  eval 60/100 | E=105.406848


| Case | Nhánh | p | Trạng thái | nfev/cap | E | P_domain % | P_valid % | P_opt % | P_ground |
|---|---|---:|---|---:|---:|---:|---:|---:|---:|
| v5x3_2 | standard_x | 1 | completed | 100/100 | 309.551864 | 29.478311 | 0.019687 | 0.019687 | 1.25052e-06 |
| v5x3_2 | standard_x | 2 | completed | 100/100 | 286.110019 | 35.374913 | 0.036670 | 0.036670 | 7.31844e-06 |
| v5x3_2 | valid_x | 1 | completed | 100/100 | 153.650814 | 61.307780 | 0.161963 | 0.161963 | 1.84433e-06 |
| v5x3_2 | valid_x | 2 | completed | 62/100 | 105.406847 | 20.594533 | 0.067334 | 0.067334 | 3.87265e-06 |
| v5x3_2 | valid_dw | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_2 | valid_dw | 2 | pending | 0/100 | - | - | - | - | - |

Completed: 10 /18 | objective: 881 / 1800
Checkpoint ZIP: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_validation\t4c_20261026T131859190531Z.zip
v5x3_2 / valid_x: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.


### v5x3_2 — valid_dw: p1 → p2


In [8]:
status = run_campaign(root, folder, instance_id='v5x3_2', variant='valid_dw')
show_status(status, 'v5x3_2')
archive = export_campaign(root,folder)
print('Checkpoint ZIP:',archive)
selected = [r for r in status['rows'] if r['instance_id']=='v5x3_2' and r['variant']=='valid_dw']
assert len(selected)==2 and all(r['status']=='completed' for r in selected), 'Stage chưa hoàn tất. Giữ ZIP/log để kiểm tra; không reset ngân sách.'
print('v5x3_2 / valid_dw: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.')


Run v5x3_2__valid_dw_p1 | 25 qubits | cap 100
  eval 1/100 | E=444.641477
  eval 10/100 | E=214.112385
  eval 20/100 | E=197.083686
  eval 30/100 | E=168.333146
  eval 40/100 | E=166.666229
  eval 50/100 | E=166.661373
Run v5x3_2__valid_dw_p2 | 25 qubits | cap 100
  eval 1/100 | E=141.330587
  eval 10/100 | E=142.398703
  eval 20/100 | E=113.338413
  eval 30/100 | E=111.732265
  eval 40/100 | E=111.217559
  eval 50/100 | E=110.617149
  eval 60/100 | E=110.230543
  eval 70/100 | E=110.114210
  eval 80/100 | E=110.085504
  eval 90/100 | E=110.050115
  eval 100/100 | E=110.026797


| Case | Nhánh | p | Trạng thái | nfev/cap | E | P_domain % | P_valid % | P_opt % | P_ground |
|---|---|---:|---|---:|---:|---:|---:|---:|---:|
| v5x3_2 | standard_x | 1 | completed | 100/100 | 309.551864 | 29.478311 | 0.019687 | 0.019687 | 1.25052e-06 |
| v5x3_2 | standard_x | 2 | completed | 100/100 | 286.110019 | 35.374913 | 0.036670 | 0.036670 | 7.31844e-06 |
| v5x3_2 | valid_x | 1 | completed | 100/100 | 153.650814 | 61.307780 | 0.161963 | 0.161963 | 1.84433e-06 |
| v5x3_2 | valid_x | 2 | completed | 62/100 | 105.406847 | 20.594533 | 0.067334 | 0.067334 | 3.87265e-06 |
| v5x3_2 | valid_dw | 1 | completed | 55/100 | 166.661366 | 100.000000 | 0.080409 | 0.080409 | 1.19969e-06 |
| v5x3_2 | valid_dw | 2 | completed | 100/100 | 110.026797 | 100.000000 | 0.210010 | 0.210010 | 9.71392e-06 |

Completed: 12 /18 | objective: 1036 / 1800
Checkpoint ZIP: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_validation\t4c_20261026T131859190531Z.zip
v5x3_2 / valid_dw: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.


## Chạy v5x3_3

Oracle có 14/243 assignment khả thi; optimum 3 host. Đây là thông tin đã khóa từ N26, chỉ dùng chấm điểm, không dùng chọn góc. Optimum=3 nên P_opt=P_valid; zero optimal shots không chứng minh bài toán vô nghiệm.

Chạy ba ô dưới theo thứ tự. Mỗi ô giữ p1 và p2 cùng một nhánh; cap200. Log xuất hiện ở eval1 và mỗi10lượt; trước eval1 phải dựng mô hình/diagonal/mạch nên có thể chờ hàng chục giây.


### v5x3_3 — standard_x: p1 → p2


In [9]:
status = run_campaign(root, folder, instance_id='v5x3_3', variant='standard_x')
show_status(status, 'v5x3_3')
archive = export_campaign(root,folder)
print('Checkpoint ZIP:',archive)
selected = [r for r in status['rows'] if r['instance_id']=='v5x3_3' and r['variant']=='standard_x']
assert len(selected)==2 and all(r['status']=='completed' for r in selected), 'Stage chưa hoàn tất. Giữ ZIP/log để kiểm tra; không reset ngân sách.'
print('v5x3_3 / standard_x: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.')


Run v5x3_3__standard_x_p1 | 25 qubits | cap 100
  eval 1/100 | E=685.440572
  eval 10/100 | E=265.854640
  eval 20/100 | E=220.926587
  eval 30/100 | E=154.067741
  eval 40/100 | E=138.265464
  eval 50/100 | E=137.995237
  eval 60/100 | E=137.990214
Run v5x3_3__standard_x_p2 | 25 qubits | cap 100
  eval 1/100 | E=159.525951
  eval 10/100 | E=117.378903
  eval 20/100 | E=90.721860
  eval 30/100 | E=89.214581
  eval 40/100 | E=87.698036
  eval 50/100 | E=87.067341
  eval 60/100 | E=86.127721
  eval 70/100 | E=85.282098
  eval 80/100 | E=85.129872
  eval 90/100 | E=84.960786
  eval 100/100 | E=84.498574


| Case | Nhánh | p | Trạng thái | nfev/cap | E | P_domain % | P_valid % | P_opt % | P_ground |
|---|---|---:|---|---:|---:|---:|---:|---:|---:|
| v5x3_3 | standard_x | 1 | completed | 61/100 | 137.990214 | 46.897493 | 1.276663 | 1.276663 | 5.95529e-06 |
| v5x3_3 | standard_x | 2 | completed | 100/100 | 84.498574 | 48.761620 | 3.045077 | 3.045077 | 4.30036e-05 |
| v5x3_3 | valid_x | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_3 | valid_x | 2 | pending | 0/100 | - | - | - | - | - |
| v5x3_3 | valid_dw | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_3 | valid_dw | 2 | pending | 0/100 | - | - | - | - | - |

Completed: 14 /18 | objective: 1197 / 1800
Checkpoint ZIP: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_validation\t4c_20261026T131859190531Z.zip
v5x3_3 / standard_x: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.


### v5x3_3 — valid_x: p1 → p2


In [10]:
status = run_campaign(root, folder, instance_id='v5x3_3', variant='valid_x')
show_status(status, 'v5x3_3')
archive = export_campaign(root,folder)
print('Checkpoint ZIP:',archive)
selected = [r for r in status['rows'] if r['instance_id']=='v5x3_3' and r['variant']=='valid_x']
assert len(selected)==2 and all(r['status']=='completed' for r in selected), 'Stage chưa hoàn tất. Giữ ZIP/log để kiểm tra; không reset ngân sách.'
print('v5x3_3 / valid_x: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.')


Run v5x3_3__valid_x_p1 | 25 qubits | cap 100
  eval 1/100 | E=456.639347
  eval 10/100 | E=230.246531
  eval 20/100 | E=201.372837
  eval 30/100 | E=176.052971
  eval 40/100 | E=166.660915
  eval 50/100 | E=151.070890
  eval 60/100 | E=140.982037
  eval 70/100 | E=134.893308
  eval 80/100 | E=134.628306
  eval 90/100 | E=134.543887
  eval 100/100 | E=134.417568
Run v5x3_3__valid_x_p2 | 25 qubits | cap 100
  eval 1/100 | E=124.631993
  eval 10/100 | E=118.094171
  eval 20/100 | E=101.468268
  eval 30/100 | E=100.163147
  eval 40/100 | E=99.186813
  eval 50/100 | E=98.120123
  eval 60/100 | E=96.255876
  eval 70/100 | E=95.593613
  eval 80/100 | E=95.157136
  eval 90/100 | E=94.873288
  eval 100/100 | E=94.684109


| Case | Nhánh | p | Trạng thái | nfev/cap | E | P_domain % | P_valid % | P_opt % | P_ground |
|---|---|---:|---|---:|---:|---:|---:|---:|---:|
| v5x3_3 | standard_x | 1 | completed | 61/100 | 137.990214 | 46.897493 | 1.276663 | 1.276663 | 5.95529e-06 |
| v5x3_3 | standard_x | 2 | completed | 100/100 | 84.498574 | 48.761620 | 3.045077 | 3.045077 | 4.30036e-05 |
| v5x3_3 | valid_x | 1 | completed | 100/100 | 134.417568 | 55.974253 | 1.857324 | 1.857324 | 1.42628e-05 |
| v5x3_3 | valid_x | 2 | completed | 100/100 | 94.684109 | 24.944772 | 1.398349 | 1.398349 | 2.99029e-05 |
| v5x3_3 | valid_dw | 1 | pending | 0/100 | - | - | - | - | - |
| v5x3_3 | valid_dw | 2 | pending | 0/100 | - | - | - | - | - |

Completed: 16 /18 | objective: 1397 / 1800
Checkpoint ZIP: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_validation\t4c_20261026T131859190531Z.zip
v5x3_3 / valid_x: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.


### v5x3_3 — valid_dw: p1 → p2


In [11]:
status = run_campaign(root, folder, instance_id='v5x3_3', variant='valid_dw')
show_status(status, 'v5x3_3')
archive = export_campaign(root,folder)
print('Checkpoint ZIP:',archive)
selected = [r for r in status['rows'] if r['instance_id']=='v5x3_3' and r['variant']=='valid_dw']
assert len(selected)==2 and all(r['status']=='completed' for r in selected), 'Stage chưa hoàn tất. Giữ ZIP/log để kiểm tra; không reset ngân sách.'
print('v5x3_3 / valid_dw: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.')


Run v5x3_3__valid_dw_p1 | 25 qubits | cap 100
  eval 1/100 | E=431.782346
  eval 10/100 | E=194.414459
  eval 20/100 | E=188.514585
  eval 30/100 | E=162.810548
  eval 40/100 | E=140.462758
  eval 50/100 | E=137.885999
  eval 60/100 | E=137.876405
Run v5x3_3__valid_dw_p2 | 25 qubits | cap 100
  eval 1/100 | E=136.832987
  eval 10/100 | E=127.875476
  eval 20/100 | E=102.140769
  eval 30/100 | E=98.205568
  eval 40/100 | E=97.939708
  eval 50/100 | E=97.807555
  eval 60/100 | E=97.784757
  eval 70/100 | E=97.769169
  eval 80/100 | E=97.739463
  eval 90/100 | E=97.733303
  eval 100/100 | E=97.723842


| Case | Nhánh | p | Trạng thái | nfev/cap | E | P_domain % | P_valid % | P_opt % | P_ground |
|---|---|---:|---|---:|---:|---:|---:|---:|---:|
| v5x3_3 | standard_x | 1 | completed | 61/100 | 137.990214 | 46.897493 | 1.276663 | 1.276663 | 5.95529e-06 |
| v5x3_3 | standard_x | 2 | completed | 100/100 | 84.498574 | 48.761620 | 3.045077 | 3.045077 | 4.30036e-05 |
| v5x3_3 | valid_x | 1 | completed | 100/100 | 134.417568 | 55.974253 | 1.857324 | 1.857324 | 1.42628e-05 |
| v5x3_3 | valid_x | 2 | completed | 100/100 | 94.684109 | 24.944772 | 1.398349 | 1.398349 | 2.99029e-05 |
| v5x3_3 | valid_dw | 1 | completed | 67/100 | 137.875948 | 100.000000 | 1.577756 | 1.577756 | 1.27182e-05 |
| v5x3_3 | valid_dw | 2 | completed | 100/100 | 97.723842 | 100.000000 | 4.209420 | 4.209420 | 5.29526e-05 |

Completed: 18 /18 | objective: 1564 / 1800
Checkpoint ZIP: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_validation\t4c_20261026T131859190531Z.zip
v5x3_3 / valid_dw: p1+p2 completed; có thể nghỉ tại đây rồi tiếp tục ô kế tiếp.


## Tổng hợp và đọc trạng thái dừng

p2 là kết quả chính, tính cả chi phí học p1. completed chỉ nghĩa thực thi/scoring hoàn tất. success=False với MAXFUN nghĩa hết cap, không phải code lỗi và không gọi là hội tụ. Không chạy thêm cho đủ cap nếu optimizer dừng sớm đúng protocol.

So sánh ba nhánh trong từng case; không so trực tiếp raw mean-energy giữa các case khác QUBO/offset/scale. P_ground bao gồm slack; P_opt giữ y đo được nhưng không yêu cầu slack đúng. Exact metrics là chính,1024shot cuối là minh họa. Cỡ mẫu kiểm chứng=3, không phải9nhánh hay18stage.


In [12]:
status = summarize(root,folder)
show_status(status)
print('COMPLETE:',status['complete'])
print('Simulation calls reserved:',status['simulation_calls_reserved'])
print('Sum stage minutes:',round(status['elapsed_stage_seconds']/60,2),'(không phải toàn bộ wall-clock campaign)')
for case in status['cases']: print(case)
for r in status['rows']:
    if 'optimizer_stop' in r: print(r['id'],r['optimizer_stop'])


| Case | Nhánh | p | Trạng thái | nfev/cap | E | P_domain % | P_valid % | P_opt % | P_ground |
|---|---|---:|---|---:|---:|---:|---:|---:|---:|
| v5x3_1 | standard_x | 1 | completed | 66/100 | 98.317334 | 45.754794 | 5.434750 | 0.207156 | 1.33952e-06 |
| v5x3_1 | standard_x | 2 | completed | 100/100 | 59.087622 | 48.518063 | 11.566711 | 0.278349 | 1.2236e-05 |
| v5x3_1 | valid_x | 1 | completed | 100/100 | 150.516848 | 72.665298 | 3.799145 | 0.297157 | 1.03636e-05 |
| v5x3_1 | valid_x | 2 | completed | 100/100 | 77.440332 | 43.550876 | 7.296463 | 0.260118 | 1.87387e-05 |
| v5x3_1 | valid_dw | 1 | completed | 53/100 | 98.814708 | 100.000000 | 13.568403 | 0.986681 | 1.90717e-05 |
| v5x3_1 | valid_dw | 2 | completed | 100/100 | 68.636648 | 100.000000 | 21.618602 | 0.848718 | 2.79811e-05 |
| v5x3_2 | standard_x | 1 | completed | 100/100 | 309.551864 | 29.478311 | 0.019687 | 0.019687 | 1.25052e-06 |
| v5x3_2 | standard_x | 2 | completed | 100/100 | 286.110019 | 35.374913 | 0.036670 | 0.036670 | 7.31844e-06 |
| v5x3_2 | valid_x | 1 | completed | 100/100 | 153.650814 | 61.307780 | 0.161963 | 0.161963 | 1.84433e-06 |
| v5x3_2 | valid_x | 2 | completed | 62/100 | 105.406847 | 20.594533 | 0.067334 | 0.067334 | 3.87265e-06 |
| v5x3_2 | valid_dw | 1 | completed | 55/100 | 166.661366 | 100.000000 | 0.080409 | 0.080409 | 1.19969e-06 |
| v5x3_2 | valid_dw | 2 | completed | 100/100 | 110.026797 | 100.000000 | 0.210010 | 0.210010 | 9.71392e-06 |
| v5x3_3 | standard_x | 1 | completed | 61/100 | 137.990214 | 46.897493 | 1.276663 | 1.276663 | 5.95529e-06 |
| v5x3_3 | standard_x | 2 | completed | 100/100 | 84.498574 | 48.761620 | 3.045077 | 3.045077 | 4.30036e-05 |
| v5x3_3 | valid_x | 1 | completed | 100/100 | 134.417568 | 55.974253 | 1.857324 | 1.857324 | 1.42628e-05 |
| v5x3_3 | valid_x | 2 | completed | 100/100 | 94.684109 | 24.944772 | 1.398349 | 1.398349 | 2.99029e-05 |
| v5x3_3 | valid_dw | 1 | completed | 67/100 | 137.875948 | 100.000000 | 1.577756 | 1.577756 | 1.27182e-05 |
| v5x3_3 | valid_dw | 2 | completed | 100/100 | 97.723842 | 100.000000 | 4.209420 | 4.209420 | 5.29526e-05 |

Completed: 18 /18 | objective: 1564 / 1800
COMPLETE: True
Simulation calls reserved: 1591
Sum stage minutes: 160.77 (không phải toàn bộ wall-clock campaign)
{'instance_id': 'v5x3_1', 'complete': True, 'completed_stages': 6, 'objective_cap': 600, 'objective_calls_reserved': 519, 'objective_calls_completed': 519, 'simulation_calls_reserved': 528, 'elapsed_stage_seconds': 2585.8175970999982}
{'instance_id': 'v5x3_2', 'complete': True, 'completed_stages': 6, 'objective_cap': 600, 'objective_calls_reserved': 517, 'objective_calls_completed': 517, 'simulation_calls_reserved': 526, 'elapsed_stage_seconds': 3890.797081500001}
{'instance_id': 'v5x3_3', 'complete': True, 'completed_stages': 6, 'objective_cap': 600, 'objective_calls_reserved': 528, 'objective_calls_completed': 528, 'simulation_calls_reserved': 537, 'elapsed_stage_seconds': 3169.462515899995}
v5x3_1__standard_x_p1 {'status': 0, 'success': True, 'message': 'Return from COBYLA because the trust region radius reaches its lower bound.

## Xuất và gửi bằng chứng

Chạy ô dưới rồi Save notebook. Gửi notebook28 đã chạy,ZIP t4c_<timestamp>.zip và N28_ACTIVE.json. Có thể xuất campaign chưa hoàn tất nếu writer đã dừng và dữ liệu còn nhất quán. Không xóa thư mục campaign/N27 trước khi kiểm toán.

Nếu kernel/PC chết cứng để lại RUNNING.lock hoặc hash lỗi, giữ nguyên thư mục, chụp nguyên lỗi và nén thư mục bằng Explorer nếu export bị chặn. Không xóa lock/raw/ledger hay đổi source để ép tiếp tục. Checkpoint mỗi evaluation không khôi phục được nội bộ COBYLA giữa stage; chỉ tiếp tục ở ranh giới stage hoàn tất.


In [13]:
archive = export_campaign(root,folder)
print('GỬI ZIP:',archive)
print('GỬI NOTEBOOK:',root/'notebooks/28_domain_wall_training_validation.ipynb')
print('GỬI POINTER:',folder.parent/'N28_ACTIVE.json')
print('Save notebook đã chạy; không cần gửi toàn bộ repo.')


GỬI ZIP: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_validation\t4c_20261026T131859190531Z.zip
GỬI NOTEBOOK: c:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\notebooks\28_domain_wall_training_validation.ipynb
GỬI POINTER: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_domainwall_validation\N28_ACTIVE.json
Save notebook đã chạy; không cần gửi toàn bộ repo.
